# Question 1 

Question 1: Install Spark and PySpark
- Install Spark
- Run PySpark
- Create a local spark session
- Execute spark.version.



What's the output?

In [2]:
from pyspark.sql import SparkSession

In [3]:
spark = (
    SparkSession.builder
        .master("local[*]")
        .appName("Test")
        .getOrCreate()
)

In [6]:
spark.version

'3.3.2'

Question 2: Yellow October 2024
Read the October 2024 Yellow into a Spark Dataframe.

Repartition the Dataframe to 4 partitions and save it to parquet.

What is the average size of the Parquet (ending with .parquet extension) Files that were created (in MB)? Select the answer which most closely matches.

- 6MB
- **25MB**
- 75MB
- 100MB

In [7]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-10.parquet

--2025-03-15 01:27:16--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-10.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 3.164.82.40, 3.164.82.112, 3.164.82.160, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|3.164.82.40|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 64346071 (61M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2024-10.parquet’

yellow_tripdata_202 100%[===================>]  61.36M  13.8MB/s    in 5.4s    

2025-03-15 01:27:22 (11.3 MB/s) - ‘yellow_tripdata_2024-10.parquet’ saved [64346071/64346071]



In [8]:
df = spark.read.parquet('yellow_tripdata_2024-10.parquet')

In [9]:
type(df)

pyspark.sql.dataframe.DataFrame

In [10]:
df = df.repartition(4)

In [11]:
!mkdir -p yellow

In [12]:
df.write.parquet('yellow/2024/10')

In [21]:
!ls -lrt yellow/2024/10

total 99244
-rw-r--r-- 1 satiy satiy 25438400 Mar 15 01:47 part-00003-12aa3503-8264-4eeb-9fc7-a5daa9d4027c-c000.snappy.parquet
-rw-r--r-- 1 satiy satiy 25375851 Mar 15 01:47 part-00002-12aa3503-8264-4eeb-9fc7-a5daa9d4027c-c000.snappy.parquet
-rw-r--r-- 1 satiy satiy 25407647 Mar 15 01:47 part-00001-12aa3503-8264-4eeb-9fc7-a5daa9d4027c-c000.snappy.parquet
-rw-r--r-- 1 satiy satiy 25394177 Mar 15 01:47 part-00000-12aa3503-8264-4eeb-9fc7-a5daa9d4027c-c000.snappy.parquet
-rw-r--r-- 1 satiy satiy        0 Mar 15 01:47 _SUCCESS


## Question 3: Count records 

How many taxi trips were there on the 15th of October?

Consider only trips that started on the 15th of October.

- 85,567
- 105,567
- **125,567**
- 145,567

In [24]:
from pyspark.sql import functions as F

In [46]:
df.count()

3833771

In [45]:
df_trans = df.withColumn('tpep_pickup_date', F.to_date(df.tpep_pickup_datetime))\
    .withColumn('tpep_dropoff_date', F.to_date(df.tpep_dropoff_datetime))\
    .select('PULocationID', 'DOLocationID', 'tpep_pickup_date', 'tpep_dropoff_date')

df_trans.filter(F.col('tpep_pickup_date')==F.lit('2024-10-15')).count()


128893

## Question 4: Longest trip

What is the length of the longest trip in the dataset in hours?

- 122
- 142
- **162**
- 182


In [49]:
df_trans = df.select('PULocationID', 'DOLocationID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime')

In [61]:
df.registerTempTable('yellow_trips')

/home/satiy/spark/spark-3.3.2-bin-hadoop3/python/pyspark/sql/dataframe.py:229: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [76]:
spark.sql("""
WITH A AS (
    SELECT 
        *, 
        TIMESTAMPDIFF(SECOND, tpep_pickup_datetime, tpep_dropoff_datetime) AS duration_seconds
    FROM yellow_trips
)
SELECT 
    *, 
    duration_seconds / 3600 AS duration_hours
FROM A
ORDER BY duration_hours desc
""").select('duration_hours').show()


+------------------+
|    duration_hours|
+------------------+
|162.61777777777777|
|           143.325|
|137.76055555555556|
|114.83472222222223|
| 89.89833333333333|
| 89.44611111111111|
| 70.29916666666666|
| 67.57333333333334|
| 66.06666666666666|
|           46.4225|
| 42.30888888888889|
| 38.47416666666667|
| 33.95111111111111|
| 26.29861111111111|
| 25.29138888888889|
|25.238333333333333|
|             24.47|
|23.996666666666666|
|23.995277777777776|
|23.994722222222222|
+------------------+
only showing top 20 rows



## Question 5: User Interface

Spark’s User Interface which shows the application's dashboard runs on which local port?

- 80
- 443
- **4040**
- 8080

## Question 6: Least frequent pickup location zone

Load the zone lookup data into a temp view in Spark:

```bash
wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
```

Using the zone lookup data and the Yellow October 2024 data, what is the name of the LEAST frequent pickup location Zone?

- **Governor's Island/Ellis Island/Liberty Island**
- Arden Heights
- Rikers Island
- Jamaica Bay


In [77]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2025-03-15 03:19:29--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 3.165.247.187, 3.165.247.79, 3.165.247.163, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|3.165.247.187|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv.1’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0.02s   

2025-03-15 03:19:30 (609 KB/s) - ‘taxi_zone_lookup.csv.1’ saved [12331/12331]



In [79]:
zones = spark.read.option('header', 'True').csv('taxi_zone_lookup.csv.1')

In [81]:
zones.registerTempTable('zones')

/home/satiy/spark/spark-3.3.2-bin-hadoop3/python/pyspark/sql/dataframe.py:229: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [82]:
zones.columns

['LocationID', 'Borough', 'Zone', 'service_zone']

In [87]:
spark.sql("""
WITH A AS
(
    SELECT
        t.PULocationID,
        dim_pu.Zone AS PUZone,
        t.DOLocationID,
        t.tpep_pickup_datetime, 
        t.tpep_dropoff_datetime
    FROM yellow_trips t
    LEFT JOIN zones dim_pu
    ON t.PULocationID=dim_pu.LocationID
)
SELECT 
    PUZone,
    count(1) AS frequency
FROM A
GROUP BY PUZone
ORDER BY count(1) asc;
""").show()


+--------------------+---------+
|              PUZone|frequency|
+--------------------+---------+
|Governor's Island...|        1|
|       Arden Heights|        2|
|       Rikers Island|        2|
|         Jamaica Bay|        3|
| Green-Wood Cemetery|        3|
|Charleston/Totten...|        4|
|       Port Richmond|        4|
|   Rossville/Woodrow|        4|
|       West Brighton|        4|
|Eltingville/Annad...|        4|
|        Crotona Park|        6|
|         Great Kills|        6|
|     Mariners Harbor|        7|
|Heartland Village...|        7|
|Saint George/New ...|        9|
|             Oakwood|        9|
|New Dorp/Midland ...|       10|
|       Broad Channel|       10|
|         Westerleigh|       12|
|     Pelham Bay Park|       12|
+--------------------+---------+
only showing top 20 rows

